# Chapter 5.7: FlashInfer Integration

## Learning Objectives

By the end of this notebook, you will:

1. **Understand FlashInfer** and its role as SGLang's primary attention backend
2. **Learn the paged KV-cache attention** API
3. **Master the Plan/Run API pattern** for efficient batch attention
4. **Understand ragged tensor support** for variable-length sequences
5. **See how SGLang configures FlashInfer** for prefill and decode
6. **Compare FlashInfer with other attention backends**

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part5/chapter_5.7_flashinfer.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part5/chapter_5.7_flashinfer.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. What is FlashInfer?

FlashInfer is a **high-performance GPU attention kernel library** designed specifically for LLM serving.

### Key Features

| Feature | Description |
|---------|-------------|
| Paged KV-Cache | Attention with paged (non-contiguous) KV-cache |
| Ragged Tensors | Variable-length sequences in one batch |
| Plan/Run API | Pre-compute metadata once, run attention multiple times |
| GQA/MQA | Native support for Grouped/Multi-Query Attention |
| FP8 KV-Cache | 8-bit quantized KV-cache support |
| CUDA Graphs | Compatible with CUDA graph capture |
| RoPE Fusion | Fused rotary position embedding |

### Why SGLang Uses FlashInfer

1. **Paged attention**: Required for RadixAttention's token-level KV management
2. **Ragged tensors**: Essential for mixing extend (variable lengths) and decode
3. **Plan/Run**: Amortizes metadata preparation across layers
4. **Performance**: Highly optimized CUDA kernels

## 2. FlashInfer's Paged KV-Cache

FlashInfer organizes KV-cache into **pages** (like virtual memory):

```
Physical KV Pages on GPU:
┌──────┬──────┬──────┬──────┬──────┬──────┬──────┬──────┐
│ P0   │ P1   │ P2   │ P3   │ P4   │ P5   │ P6   │ P7   │
│(free)│ K,V  │ K,V  │(free)│ K,V  │ K,V  │ K,V  │(free)│
└──────┴──────┴──────┴──────┴──────┴──────┴──────┴──────┘

Page Tables (per sequence):
Seq A: [P1, P2, P5]        -> KV at pages 1, 2, 5
Seq B: [P4, P6]            -> KV at pages 4, 6

FlashInfer reads the page table to find KV-cache:
  For Seq A, token at position 5 in page P2:
    physical_addr = kv_data[P2][position_in_page]
```

### Page Size

SGLang typically uses **page_size=1** (each page = 1 token) for RadixAttention.
This gives token-level granularity for cache management.

```python
# FlashInfer KV-cache layout
# kv_data shape: [max_pages, 2, num_kv_heads, page_size, head_dim]
#                  │         │   │             │          └─ per-head dim
#                  │         │   │             └─ tokens per page
#                  │         │   └─ GQA groups
#                  │         └─ K and V
#                  └─ pre-allocated pages
```

In [ ]:
# Simulate FlashInfer's paged KV-cache

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
import random

class PagedKVCache:
    """Simulates FlashInfer's paged KV-cache."""
    
    def __init__(self, max_pages: int, page_size: int, num_layers: int, 
                 num_kv_heads: int, head_dim: int):
        self.max_pages = max_pages
        self.page_size = page_size
        self.num_layers = num_layers
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        
        # Simulated KV data (just track allocation)
        self.page_allocated = [False] * max_pages
        self.free_pages = list(range(max_pages))
        
        # Page tables per sequence
        self.page_tables: Dict[str, List[int]] = {}  # seq_id -> [page_ids]
    
    def allocate_pages(self, seq_id: str, num_tokens: int) -> List[int]:
        """Allocate pages for a sequence."""
        num_pages = (num_tokens + self.page_size - 1) // self.page_size
        
        if len(self.free_pages) < num_pages:
            raise MemoryError(f"Need {num_pages} pages, only {len(self.free_pages)} free")
        
        allocated = []
        for _ in range(num_pages):
            page_id = self.free_pages.pop(0)
            self.page_allocated[page_id] = True
            allocated.append(page_id)
        
        if seq_id in self.page_tables:
            self.page_tables[seq_id].extend(allocated)
        else:
            self.page_tables[seq_id] = allocated
        
        return allocated
    
    def free_pages_for_seq(self, seq_id: str):
        """Free all pages for a sequence."""
        if seq_id in self.page_tables:
            for page_id in self.page_tables[seq_id]:
                self.page_allocated[page_id] = False
                self.free_pages.append(page_id)
            del self.page_tables[seq_id]
    
    def get_page_table(self, seq_id: str) -> List[int]:
        return self.page_tables.get(seq_id, [])
    
    def memory_bytes_per_page(self):
        # 2 (K+V) * num_layers * num_kv_heads * page_size * head_dim * 2 (FP16)
        return 2 * self.num_layers * self.num_kv_heads * self.page_size * self.head_dim * 2
    
    def display(self):
        total = self.max_pages
        used = sum(1 for p in self.page_allocated if p)
        print(f"\nPaged KV-Cache Status")
        print(f"  Total pages: {total}, Used: {used}, Free: {total - used}")
        print(f"  Page size: {self.page_size} token(s)")
        print(f"  Memory per page: {self.memory_bytes_per_page():,} bytes")
        
        # Visual
        row_size = 20
        print(f"\n  Page map (. = free, # = used):")
        for i in range(0, min(total, 100), row_size):
            row = ""
            for j in range(row_size):
                if i + j < total:
                    row += "#" if self.page_allocated[i+j] else "."
            print(f"    [{i:3d}-{min(i+row_size-1, total-1):3d}] {row}")
        
        if self.page_tables:
            print(f"\n  Page tables:")
            for seq_id, pages in self.page_tables.items():
                tokens = len(pages) * self.page_size
                print(f"    {seq_id}: pages={pages[:10]}{'...' if len(pages) > 10 else ''} ({tokens} tokens)")

# Demo
cache = PagedKVCache(
    max_pages=100, page_size=1,  # SGLang default: 1 token per page
    num_layers=32, num_kv_heads=8, head_dim=128
)

# Allocate for several sequences
cache.allocate_pages("seq_A", 50)   # 50 tokens
cache.allocate_pages("seq_B", 30)   # 30 tokens
cache.allocate_pages("seq_C", 15)   # 15 tokens

cache.display()

# Free one sequence
print("\nFreeing seq_B...")
cache.free_pages_for_seq("seq_B")

# Allocate new sequence (will fill gaps)
cache.allocate_pages("seq_D", 20)
cache.display()

## 3. The Plan/Run API Pattern

FlashInfer's key innovation is the **Plan/Run** pattern:

```python
# Traditional approach: prepare metadata in every attention layer
for layer in model.layers:
    # Recompute metadata EVERY layer (wasteful!)
    metadata = prepare_attention_metadata(batch)
    output = flash_attention(q, k, v, metadata)

# FlashInfer Plan/Run: prepare once, run many times
# PLAN phase (once per forward pass)
wrapper.plan(
    qo_indptr=qo_indptr,         # Query offsets
    paged_kv_indptr=kv_indptr,   # KV page table offsets
    paged_kv_indices=kv_indices, # KV page indices
    paged_kv_last_page_len=last_page_len,
    num_qo_heads=32,
    num_kv_heads=8,
    head_dim=128,
)

# RUN phase (once per layer, reuses planned metadata)
for layer in model.layers:
    output = wrapper.run(q, kv_data)  # Much faster!
```

### Why Plan/Run?

1. **Metadata preparation** involves sorting, partitioning, and GPU memory allocation
2. **All layers share the same batch structure** (same sequences, same KV pages)
3. **Plan** does this work once; **Run** just executes the attention kernel
4. Reduces overhead from O(num_layers) to O(1) for metadata preparation

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatches# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, ax = plt.subplots(1, 1, figsize=(14, 5))ax.set_xlim(0, 14)ax.set_ylim(0, 6)ax.axis('off')fig.patch.set_facecolor('white')ax.text(7, 5.5, 'FlashInfer Plan/Run Pattern', fontsize=14, fontweight='bold',        ha='center', color=DARK_GRAY)# Plan phaseplan_box = mpatches.FancyBboxPatch((0.5, 2), 4.5, 2.5, boxstyle="round,pad=0.15",                                    facecolor=BLUE, alpha=0.85, edgecolor='white', lw=2)ax.add_patch(plan_box)ax.text(2.75, 3.9, 'PLAN Phase (CPU)', fontsize=11, fontweight='bold', color='white', ha='center')plan_items = ['Build page table indices', 'Compute access patterns', 'Allocate workspace']for i, item in enumerate(plan_items):    ax.text(2.75, 3.1 - i*0.4, f'  {item}', fontsize=8, color='white', ha='center')# Arrowax.annotate('', xy=(6.2, 3.25), xytext=(5.2, 3.25),            arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=3))ax.text(5.7, 3.7, 'Once per\nforward pass', fontsize=8, color=DARK_GRAY, ha='center')# Run phaserun_box = mpatches.FancyBboxPatch((6.5, 2), 7, 2.5, boxstyle="round,pad=0.15",                                   facecolor=GREEN, alpha=0.85, edgecolor='white', lw=2)ax.add_patch(run_box)ax.text(10, 3.9, 'RUN Phase (GPU) -- Once per Layer', fontsize=11, fontweight='bold', color='white', ha='center')# Layer boxesfor i in range(4):    lx = 7 + i * 1.6    layer_box = mpatches.FancyBboxPatch((lx, 2.3), 1.3, 1.2, boxstyle="round,pad=0.05",                                        facecolor='white', alpha=0.3, edgecolor='white', lw=1)    ax.add_patch(layer_box)    ax.text(lx + 0.65, 2.9, f'Layer {i}', fontsize=8, color='white', ha='center', fontweight='bold')ax.text(12.5, 2.9, '...', fontsize=14, color='white', ha='center')# Metadata reuse annotationax.text(10, 1.2, 'Metadata computed ONCE, reused across all 32+ layers',        fontsize=10, ha='center', color=DARK_GRAY, fontweight='bold',        bbox=dict(boxstyle='round,pad=0.3', facecolor='#F0F0F0', alpha=0.8))plt.tight_layout()plt.savefig("flashinfer_plan_run.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Demonstrate the Plan/Run API

class FlashInferWrapper:
    """Simulates FlashInfer's BatchPrefillWithPagedKVCache."""
    
    def __init__(self, name="prefill"):
        self.name = name
        self.planned = False
        self.plan_metadata = None
        self.run_count = 0
    
    def plan(self, qo_indptr, kv_indptr, kv_indices, kv_last_page_len,
             num_qo_heads, num_kv_heads, head_dim, page_size=1):
        """Plan phase: prepare metadata for attention computation."""
        self.plan_metadata = {
            "num_sequences": len(qo_indptr) - 1,
            "qo_indptr": qo_indptr,
            "kv_indptr": kv_indptr,
            "kv_indices": kv_indices,
            "num_qo_heads": num_qo_heads,
            "num_kv_heads": num_kv_heads,
            "head_dim": head_dim,
            "gqa_ratio": num_qo_heads // num_kv_heads,
        }
        self.planned = True
        self.run_count = 0
        
        # Calculate compute
        total_q_tokens = qo_indptr[-1]
        total_kv_pages = len(kv_indices)
        
        print(f"  [{self.name}] PLAN:")
        print(f"    Sequences: {self.plan_metadata['num_sequences']}")
        print(f"    Total query tokens: {total_q_tokens}")
        print(f"    Total KV pages: {total_kv_pages}")
        print(f"    GQA ratio: {self.plan_metadata['gqa_ratio']}x")
    
    def run(self, q_data, kv_data):
        """Run phase: execute attention with pre-planned metadata."""
        if not self.planned:
            raise RuntimeError("Must call plan() before run()")
        
        self.run_count += 1
        
        # In reality, this launches a highly optimized CUDA kernel
        meta = self.plan_metadata
        total_q = meta['qo_indptr'][-1]
        
        # Simulate output
        output_shape = f"[{total_q}, {meta['num_qo_heads']}, {meta['head_dim']}]"
        return f"attention_output{output_shape}"

# Demo: Plan once, run for each layer
print("FlashInfer Plan/Run Demo")
print("=" * 60)

# Batch with 3 sequences
# Seq 0: 50 query tokens, 100 KV tokens (50 cached + 50 new)
# Seq 1: 30 query tokens, 30 KV tokens (no cache)
# Seq 2: 20 query tokens, 80 KV tokens (60 cached + 20 new)

wrapper = FlashInferWrapper("BatchPrefill")

# Plan
wrapper.plan(
    qo_indptr=[0, 50, 80, 100],     # Query offsets: [0, 50), [50, 80), [80, 100)
    kv_indptr=[0, 100, 130, 210],   # KV page offsets
    kv_indices=list(range(210)),      # KV page indices (simplified)
    kv_last_page_len=[1, 1, 1],     # All pages full (page_size=1)
    num_qo_heads=32,
    num_kv_heads=8,
    head_dim=128,
)

# Run for each layer
print(f"\n  Running attention for 32 layers:")
for layer_idx in range(32):
    output = wrapper.run(f"q_layer{layer_idx}", "kv_data")

print(f"    Completed {wrapper.run_count} runs with 1 plan")
print(f"    Metadata reuse: {wrapper.run_count}x")

## 4. BatchPrefillWithPagedKVCache vs BatchDecodeWithPagedKVCache

FlashInfer provides two main kernels:

### BatchPrefillWithPagedKVCache
- Used for **extend (prefill)** mode
- Each sequence can have **multiple query tokens**
- Handles ragged (variable-length) sequences
- Supports causal masking within each sequence

### BatchDecodeWithPagedKVCache
- Used for **decode** mode
- Each sequence has **exactly 1 query token**
- Optimized for the "tall-skinny" attention pattern
- Higher throughput per token for decode

```python
# sglang/srt/layers/attention/flashinfer_backend.py (simplified)

class FlashInferAttnBackend:
    def __init__(self, model_config, server_args):
        # Create wrappers for both modes
        self.prefill_wrapper = BatchPrefillWithPagedKVCacheWrapper(
            workspace_buffer, "NHD"  # layout
        )
        self.decode_wrapper = BatchDecodeWithPagedKVCacheWrapper(
            workspace_buffer, "NHD"
        )
    
    def init_forward_metadata(self, batch, forward_mode):
        """Initialize attention metadata (PLAN phase)."""
        if forward_mode == ForwardMode.EXTEND:
            self.prefill_wrapper.plan(
                qo_indptr=batch.extend_prefix_lens_cumsum,
                paged_kv_indptr=batch.kv_indptr,
                paged_kv_indices=batch.kv_indices,
                ...
            )
        else:  # DECODE
            self.decode_wrapper.plan(
                indptr=batch.kv_indptr,
                indices=batch.kv_indices,
                ...
            )
    
    def forward(self, q, k, v, batch, layer_id):
        """Execute attention (RUN phase)."""
        # Store K, V into paged cache
        self._store_kv(k, v, batch, layer_id)
        
        # Run attention
        if batch.forward_mode == ForwardMode.EXTEND:
            output = self.prefill_wrapper.run(q, self.kv_data)
        else:
            output = self.decode_wrapper.run(q, self.kv_data)
        
        return output
```

In [ ]:
# Compare prefill vs decode attention patterns

def analyze_attention_compute(mode, batch_info):
    """Analyze attention computation for a batch."""
    print(f"\n{mode} Mode Attention Analysis")
    print("=" * 60)
    
    total_q_tokens = 0
    total_kv_tokens = 0
    total_flops = 0
    
    num_q_heads = 32
    num_kv_heads = 8
    head_dim = 128
    gqa_ratio = num_q_heads // num_kv_heads
    
    for seq in batch_info:
        q_tokens = seq["q_tokens"]
        kv_tokens = seq["kv_tokens"]
        
        total_q_tokens += q_tokens
        total_kv_tokens += kv_tokens
        
        # FLOPs per head: 2 * q_tokens * kv_tokens * head_dim (QK^T + softmax*V)
        flops = 2 * q_tokens * kv_tokens * head_dim * num_q_heads
        total_flops += flops
        
        print(f"  Seq {seq['id']}: Q={q_tokens}, KV={kv_tokens}, "
              f"FLOPs={flops/1e9:.2f} GFLOPs")
    
    print(f"\n  Total: Q={total_q_tokens}, KV={total_kv_tokens}")
    print(f"  Total FLOPs: {total_flops/1e9:.2f} GFLOPs")
    print(f"  Arithmetic intensity: {total_flops / (total_kv_tokens * num_kv_heads * head_dim * 2 * 2):.1f} FLOPs/byte")
    
    return total_flops

# Extend (prefill) batch
extend_batch = [
    {"id": "A", "q_tokens": 100, "kv_tokens": 200},  # Long prefill
    {"id": "B", "q_tokens": 50, "kv_tokens": 50},     # Short prefill
    {"id": "C", "q_tokens": 30, "kv_tokens": 150},    # Cached prefix
]
extend_flops = analyze_attention_compute("EXTEND (Prefill)", extend_batch)

# Decode batch
decode_batch = [
    {"id": "D", "q_tokens": 1, "kv_tokens": 200},
    {"id": "E", "q_tokens": 1, "kv_tokens": 500},
    {"id": "F", "q_tokens": 1, "kv_tokens": 150},
    {"id": "G", "q_tokens": 1, "kv_tokens": 300},
    {"id": "H", "q_tokens": 1, "kv_tokens": 400},
]
decode_flops = analyze_attention_compute("DECODE", decode_batch)

print(f"\nCompute ratio: Extend/Decode = {extend_flops/decode_flops:.1f}x")

## 5. Ragged Tensor Support

In a batch, sequences have **different lengths**. FlashInfer handles this with **indptr** (index pointer) arrays:

```
3 sequences with lengths [5, 3, 7]:

Flattened tokens: [t0, t1, t2, t3, t4, t5, t6, t7, t8, t9, t10, t11, t12, t13, t14]
                   └─── Seq 0 ────┘  └─ Seq 1─┘  └──── Seq 2 ──────────────────┘

qo_indptr: [0, 5, 8, 15]
           │  │  │  └─ end of seq 2 (total tokens)
           │  │  └─ start of seq 2 / end of seq 1
           │  └─ start of seq 1 / end of seq 0
           └─ start of seq 0

To get tokens for sequence i:
  start = qo_indptr[i]
  end = qo_indptr[i+1]
  tokens = flattened[start:end]
```

In [ ]:
# Demonstrate ragged tensor representation

class RaggedTensor:
    """Simulates ragged tensor using indptr."""
    
    def __init__(self, sequences: List[List[int]]):
        self.data = []
        self.indptr = [0]
        
        for seq in sequences:
            self.data.extend(seq)
            self.indptr.append(len(self.data))
    
    @property
    def num_sequences(self):
        return len(self.indptr) - 1
    
    @property
    def total_tokens(self):
        return len(self.data)
    
    def get_sequence(self, idx: int) -> List[int]:
        start = self.indptr[idx]
        end = self.indptr[idx + 1]
        return self.data[start:end]
    
    def get_lengths(self) -> List[int]:
        return [self.indptr[i+1] - self.indptr[i] for i in range(self.num_sequences)]
    
    def display(self):
        print(f"Ragged Tensor: {self.num_sequences} sequences, {self.total_tokens} total tokens")
        print(f"  indptr: {self.indptr}")
        print(f"  lengths: {self.get_lengths()}")
        print(f"  data: {self.data[:30]}{'...' if len(self.data) > 30 else ''}")
        print()
        for i in range(self.num_sequences):
            seq = self.get_sequence(i)
            seq_str = str(seq[:10]) + ("..." if len(seq) > 10 else "")
            print(f"  Seq {i}: [{self.indptr[i]:3d}, {self.indptr[i+1]:3d}) = {seq_str}")

# Demo
sequences = [
    list(range(100, 108)),    # 8 tokens
    list(range(200, 203)),    # 3 tokens
    list(range(300, 312)),    # 12 tokens
    list(range(400, 405)),    # 5 tokens
]

ragged = RaggedTensor(sequences)
ragged.display()

print("\nThis is how FlashInfer represents variable-length batches!")
print("The indptr array lets it find each sequence without padding.")

## 6. SGLang's FlashInfer Configuration

```python
# sglang/srt/layers/attention/flashinfer_backend.py (key parts)

class FlashInferAttnBackend:
    """FlashInfer attention backend for SGLang."""
    
    def __init__(self, model_config, server_args):
        import flashinfer
        
        self.num_qo_heads = model_config.num_attention_heads // server_args.tp_size
        self.num_kv_heads = model_config.num_key_value_heads // server_args.tp_size
        self.head_dim = model_config.head_dim
        self.page_size = 1  # Token-level paging for RadixAttention
        
        # Allocate workspace for FlashInfer
        workspace_size = 128 * 1024 * 1024  # 128 MB
        self.workspace = torch.empty(
            workspace_size, dtype=torch.uint8, device='cuda'
        )
        
        # Create wrappers
        self.prefill_wrapper = flashinfer.BatchPrefillWithPagedKVCacheWrapper(
            self.workspace, "NHD"  # [seq_len, num_heads, head_dim]
        )
        self.decode_wrapper = flashinfer.BatchDecodeWithPagedKVCacheWrapper(
            self.workspace, "NHD"
        )
    
    def init_forward_metadata_for_extend(self, batch):
        """Plan for extend (prefill) mode."""
        self.prefill_wrapper.plan(
            qo_indptr=batch.qo_indptr,           # Query token boundaries
            paged_kv_indptr=batch.kv_indptr,      # KV page boundaries
            paged_kv_indices=batch.kv_page_indices, # Actual page IDs
            paged_kv_last_page_len=batch.kv_last_page_len,
            num_qo_heads=self.num_qo_heads,
            num_kv_heads=self.num_kv_heads,
            head_dim=self.head_dim,
            page_size=self.page_size,
            causal=True,
        )
    
    def init_forward_metadata_for_decode(self, batch):
        """Plan for decode mode."""
        self.decode_wrapper.plan(
            indptr=batch.kv_indptr,
            indices=batch.kv_page_indices,
            last_page_len=batch.kv_last_page_len,
            num_qo_heads=self.num_qo_heads,
            num_kv_heads=self.num_kv_heads,
            head_dim=self.head_dim,
            page_size=self.page_size,
        )
```

In [ ]:
# Performance comparison simulation

def estimate_attention_performance(mode, batch_size, seq_len, num_heads, head_dim):
    """Estimate attention performance characteristics."""
    if mode == "prefill":
        # Compute-bound: O(seq_len^2 * head_dim * num_heads)
        flops = 2 * batch_size * seq_len * seq_len * head_dim * num_heads
        # Memory: read Q, K, V + write O
        memory_bytes = batch_size * seq_len * num_heads * head_dim * 2 * 4  # Q,K,V,O
    else:
        # Memory-bound: O(seq_len * head_dim * num_heads * batch_size)
        flops = 2 * batch_size * 1 * seq_len * head_dim * num_heads
        # Memory: read full KV-cache for each sequence
        memory_bytes = batch_size * seq_len * num_heads * head_dim * 2 * 2  # K,V read
    
    # A100 specs
    peak_tflops = 312  # FP16 TFLOPS
    peak_bandwidth_tb = 2.0  # TB/s
    
    compute_time_ms = flops / (peak_tflops * 1e12) * 1000
    memory_time_ms = memory_bytes / (peak_bandwidth_tb * 1e12) * 1000
    
    bottleneck = "compute" if compute_time_ms > memory_time_ms else "memory"
    
    return {
        "mode": mode,
        "flops_gf": flops / 1e9,
        "memory_mb": memory_bytes / 1e6,
        "compute_ms": compute_time_ms,
        "memory_ms": memory_time_ms,
        "estimated_ms": max(compute_time_ms, memory_time_ms),
        "bottleneck": bottleneck,
    }

print("Attention Performance Analysis (A100 80GB)")
print("=" * 80)
print(f"{'Mode':<10s} {'BS':>4s} {'SeqLen':>8s} {'GFLOPs':>10s} {'Mem MB':>8s} "
      f"{'Time ms':>9s} {'Bottleneck':>12s}")
print("-" * 80)

scenarios = [
    ("prefill", 1, 2048),
    ("prefill", 4, 512),
    ("prefill", 1, 8192),
    ("decode", 1, 2048),
    ("decode", 16, 2048),
    ("decode", 64, 2048),
    ("decode", 128, 4096),
]

for mode, bs, seq_len in scenarios:
    result = estimate_attention_performance(mode, bs, seq_len, 32, 128)
    print(f"{result['mode']:<10s} {bs:4d} {seq_len:8d} {result['flops_gf']:10.1f} "
          f"{result['memory_mb']:8.1f} {result['estimated_ms']:9.3f} {result['bottleneck']:>12s}")

## 7. Comparison with Other Backends

| Feature | FlashInfer | FlashAttention-2 | Triton |
|---------|-----------|-------------------|--------|
| Paged KV-cache | Native | Via wrapper | Custom |
| Ragged tensors | Native | Requires padding | Custom |
| Plan/Run API | Yes | No | No |
| Decode optimization | Dedicated kernel | Same kernel | Custom |
| GQA/MQA | Native | Native | Custom |
| FP8 KV-cache | Yes | No | Custom |
| CUDA graph compat | Yes | Yes | Yes |
| Used by | SGLang | vLLM | Both |

## 8. Summary

### Key Takeaways

1. **FlashInfer** is SGLang's primary attention backend, providing paged KV-cache and ragged tensor support
2. **Plan/Run API** separates metadata preparation from kernel execution, amortizing overhead across layers
3. **Two kernels**: `BatchPrefill` for extend mode, `BatchDecode` for decode mode
4. **Page size = 1** in SGLang for token-level granularity matching RadixAttention
5. **Ragged tensors** via indptr arrays eliminate padding waste
6. **Decode is memory-bound**, prefill is compute-bound

### Next Chapter

Chapter 5.8 will explore **Constrained Decoding** — JSON, regex, and grammar-guided generation.

## Exercises

### Exercise 1: Implement indptr Construction
Write a function that takes a list of sequence lengths and constructs the indptr, kv_indptr, and kv_indices arrays for FlashInfer.

### Exercise 2: Memory Analysis
Calculate the workspace memory needed for FlashInfer given:
- Max batch size = 128
- Max sequence length = 8192
- Page size = 1

### Exercise 3: Benchmark Design
Design a benchmark that compares FlashInfer prefill vs decode performance across different batch sizes and sequence lengths. Plot the results.

In [ ]:
# Exercise 1 starter

def build_flashinfer_metadata(seq_lens, kv_page_tables, page_size=1):
    """Build FlashInfer metadata from sequence information.
    
    Args:
        seq_lens: List of query lengths per sequence
        kv_page_tables: List of page ID lists per sequence
        page_size: Tokens per page
    
    Returns:
        qo_indptr, kv_indptr, kv_indices, kv_last_page_len
    """
    # TODO: Implement
    # qo_indptr = cumulative sum of query lengths, starting with 0
    # kv_indptr = cumulative sum of KV page counts, starting with 0
    # kv_indices = flattened page IDs
    # kv_last_page_len = tokens in the last page of each sequence
    pass

print("Exercise 1: Implement build_flashinfer_metadata()")